In [ ]:
import numpy as np

In [ ]:
### some configs
len_fnl = 10
fnl_lo_arr = np.logspace(0, 5, len_fnl, base=10)

sky = 'N'
rebin = 's1'

GaussCov = 0

do_rotation = 1

mixmat_type = 'PcB'

save_result = 1
Bonly = 0
with_SN = 1 # with SN in the theory. It is okay, since best fit a2 will be close to -1 
weight = 'fkp'

if do_rotation == 0 or rebin != 's1':
    mixmat_tag = ''
elif do_rotation == 1 and rebin == 's1':
    mixmat_tag = '_%s' %mixmat_type

GaussCov = 0 # use the numerical covariance

if weight == 'fkp':
    weight_str = ''
elif weight == 'NN':
    weight_str = '_NNcat'

In [ ]:
# Load data

P0_dat = np.loadtxt('P0_dat.dat', unpack=True) 
B0_dat = np.loadtxt('B0_dat.dat', unpack=True) 
P0B0_dat = np.hstack([P0_dat, B0_dat])

# copy the not rotated data
P0B0_dat_norot = P0B0_dat*1
B0_dat_norot = B0_dat*1

In [ ]:
# Compute the cosine

SNR_dat = np.zeros((len_fnl, len(P0B0_dat)-1))
SNR_th = np.zeros((len_fnl, len(P0B0_dat)-1))
SNR_th_dat = np.zeros((len_fnl, len(P0B0_dat)-1))

for i_fnl in range(len(fnl_lo_arr)):

    fnl_lo = fnl_lo_arr[i_fnl]

    P0B0_th = computeP0B0_th(fnl_lo) # need function to compute P0+B0, as a function fnl_lo

    if do_rotation and rebin == 's1':

        # load the rotation matrix
        fname_mat = 'mixmat_matrix.dat'
        mixmat = np.loadtxt(fname_mat)
            
        # compute the covariance
        covP0B0 = np.loadtxt('covP0B0.dat', unpack=True) # load the covariance of P0+B0
        covP0B0_norot = covP0B0*1 # copy the not rotated covariance first

        # perform the rotation
        P0_th= P0B0_th[0:len(P0_dat)]
        B0_th = P0B0_th[len(P0_dat):]

        B0_th = np.dot(mixmat, B0_th)
        B0_dat = np.dot(mixmat, B0_dat_norot)
        
        P0B0_th = np.concatenate([P0_th, B0_th])
        P0B0_dat = np.concatenate([P0_dat, B0_dat])

    SNR_dat_n = []
    SNR_th_n = []
    SNR_th_dat_n = []

    for n in range(1, len(P0B0_th)):
        num_mocks = 1000 # number of mocks for numerical covariance
        hartlap_factor = (num_mocks - n - 2.) / (num_mocks - 1.)

        if do_rotation == 0:
            SNR_dat_n.append(np.einsum('i, ij, j -> ', P0B0_dat_norot[0:n], hartlap_factor*np.linalg.inv(covP0B0_norot[0:n, 0:n]), P0B0_dat_norot[0:n]))
        elif do_rotation == 1:
            SNR_dat_n.append(np.einsum('i, ij, j -> ', P0B0_dat[0:n], hartlap_factor*np.linalg.inv(covP0B0[0:n, 0:n]), P0B0_dat[0:n]))

        SNR_th_n.append(np.einsum('i, ij, j -> ', P0B0_th[0:n], hartlap_factor*np.linalg.inv(covP0B0[0:n, 0:n]), P0B0_th[0:n]))
        SNR_th_dat_n.append(np.einsum('i, ij, j -> ', P0B0_th[0:n], hartlap_factor*np.linalg.inv(covP0B0[0:n, 0:n]), P0B0_dat[0:n]))
    
    SNR_dat[i_fnl, :] = np.array([SNR_dat_n])
    SNR_th[i_fnl, :] = np.array([SNR_th_n])
    SNR_th_dat[i_fnl, :] = np.array([SNR_th_dat_n])
    
SNR_cos = np.array(SNR_th_dat)/np.sqrt(np.array(SNR_th)*np.array(SNR_dat))
fudge_fact = np.array(SNR_th_dat)/np.array(SNR_th)